# Temporary Laya server for the Kravel demo

This notebook starts the open Laya System-1 classifier on free Colab CPU and creates an authenticated HTTPS quick tunnel for one disposable Killercoda session. Use synthetic demo evidence only. The URL and bearer token are generated at runtime, are not saved in the Kravel repository, and stop working when this Colab runtime ends. **Do not save or share the notebook output.**

In [ ]:
import hashlib, os, pathlib, subprocess, sys, urllib.request

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "laya[serve]==0.3.20"], check=True)
cloudflared_path = pathlib.Path("/tmp/cloudflared")
urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/download/2026.9.3/cloudflared-linux-amd64", cloudflared_path)
actual_sha256 = hashlib.sha256(cloudflared_path.read_bytes()).hexdigest()
expected_sha256 = "53b7a7a5420d188758d24341294acb0d1bca54296548ac05e38811a694ac6134"
if actual_sha256 != expected_sha256:
    raise RuntimeError("cloudflared checksum verification failed")
os.chmod(cloudflared_path, 0o700)
print("Laya and the temporary tunnel helper are installed.")

In [ ]:
import os, re, secrets, subprocess, time
from urllib.request import urlopen

for process_name in ("laya_process", "tunnel_process"):
    previous = globals().get(process_name)
    if previous is not None and previous.poll() is None:
        previous.terminate()

laya_token = secrets.token_urlsafe(32)
environment = os.environ.copy()
environment.update({
    "LAYA_HOST": "127.0.0.1",
    "LAYA_PORT": "8000",
    "LAYA_DEVICE": "cpu",
    "LAYA_PRELOAD": "1",
    "LAYA_MODELS": "english",
    "LAYA_THREADS": "2",
    "LAYA_API_KEY": laya_token,
})
laya_log = open("/tmp/kravel-laya.log", "w")
laya_process = subprocess.Popen(["laya-serve"], env=environment, stdout=laya_log, stderr=subprocess.STDOUT)

for _ in range(180):
    if laya_process.poll() is not None:
        raise RuntimeError("Laya stopped during startup; inspect /tmp/kravel-laya.log")
    try:
        urlopen("http://127.0.0.1:8000/health", timeout=2).read()
        break
    except Exception:
        time.sleep(2)
else:
    raise TimeoutError("Laya did not become healthy within six minutes")

tunnel_process = subprocess.Popen(
    ["/tmp/cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
public_url = None
for _ in range(120):
    line = tunnel_process.stdout.readline()
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break
    if tunnel_process.poll() is not None:
        raise RuntimeError("The temporary HTTPS tunnel stopped during startup")

if not public_url:
    raise TimeoutError("No temporary HTTPS URL was issued")

print("Laya is ready. Enter these values only when compare-flows.sh prompts for them:")
print("URL:  " + public_url + "/v1/systemone")
print("Token: " + laya_token)
print("Keep this runtime open. Do not save or share this output.")